# 🛡️ Face Anti-Spoofing — 16: 웹캠 도메인 갭 해결 Fine-tuning v2
> AI Security & Application · 단국대학교 소프트웨어학과  
> 학번: 32214391 · 조현수

---

## 🎯 목표
- 기존 시도 3 실패 원인: unfreeze 30% 과다 → spoof head 과적합
- **이번 전략: unfreeze 15% + 균형 잡힌 데이터 구성**

## 📦 학습 데이터 구성
| 카테고리 | 출처 | 장수 | 역할 |
|---------|------|------|------|
| webcam_live (LCC) | LCC FASD | 200장 | 중간 도메인 브릿지 |
| webcam_live (본인) | 직접 촬영 | 51장 | 고Laplacian 웹캠 대표 |
| CelebA Live | CelebA-Spoof | 251장 | 기존 도메인 망각 방지 (1:1) |
| CelebA Print | CelebA-Spoof | 300장 | 공격 유형 유지 |
| CelebA Replay | CelebA-Spoof | 300장 | 공격 유형 유지 |
| CelebA Mask | CelebA-Spoof | 300장 | 공격 유형 유지 |

## ✅ 체크리스트
- [ ] Cell 1: Drive 마운트 + 경로 확인
- [ ] Cell 2: 데이터 로드 & 구성
- [ ] Cell 3: 데이터 분포 시각화
- [ ] Cell 4: Phase A — Head만 재학습 (backbone 동결)
- [ ] Cell 5: Phase B — 상위 15% unfreeze Fine-tuning
- [ ] Cell 6: 검증 — 카테고리별 정확도
- [ ] Cell 7: 모순 케이스 (FAKE + spoof_type=Live) 확인
- [ ] Cell 8: 모델 저장

## Cell 1 — Drive 마운트 + 경로 확인

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import tensorflow as tf
print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

BASE        = '/content/drive/MyDrive/face-anti-spoofing'
CROP_DIR    = f'{BASE}/data/cropped'      # live / print / replay / mask
WEBCAM_DIR  = f'{BASE}/data/webcam_live'  # LCC FASD Real (15_에서 저장)
MY_CAM_DIR  = f'{BASE}/data/my_webcam'    # 본인 웹캠 51장 (기존)
MODEL_DIR   = f'{BASE}/models'
REPORT_DIR  = f'{BASE}/reports'

print('\n=== 데이터 현황 ===')
dirs = {
    'CelebA live'  : f'{CROP_DIR}/live',
    'CelebA print' : f'{CROP_DIR}/print',
    'CelebA replay': f'{CROP_DIR}/replay',
    'CelebA mask'  : f'{CROP_DIR}/mask',
    'LCC webcam'   : WEBCAM_DIR,
    'My webcam'    : MY_CAM_DIR,
}
for name, d in dirs.items():
    n = len([f for f in os.listdir(d) if f.endswith(('.jpg','.jpeg','.png'))]) if os.path.exists(d) else 0
    print(f'  {name:<15}: {n}장')

print('\n=== 모델 현황 ===')
for f in os.listdir(MODEL_DIR):
    if f.endswith('.h5'):
        size = os.path.getsize(f'{MODEL_DIR}/{f}') / 1024 / 1024
        print(f'  {f:<30}: {size:.1f} MB')

## Cell 2 — 데이터 로드 & 구성

> **핵심 전략:** webcam Live 총 251장 = LCC 200 + 본인 51  
> CelebA Live도 동수(251장)로 샘플링 → **1:1 균형**으로 도메인 신호 강화

In [ ]:
import cv2
import numpy as np
import random
from pathlib import Path
from sklearn.model_selection import train_test_split

random.seed(42)
np.random.seed(42)

IMG_SIZE = 224

def load_img(path):
    raw = np.fromfile(str(path), dtype=np.uint8)
    img = cv2.imdecode(raw, cv2.IMREAD_COLOR)
    if img is None:
        return None
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img.astype(np.float32) / 255.0

def load_from_dir(d, max_n=None, label_bin=0, label_spoof=0):
    """디렉토리에서 이미지 로드. (img, binary_label, spoof_label) 반환"""
    paths = sorted(Path(d).glob('*.jpg')) + sorted(Path(d).glob('*.png'))
    if max_n:
        paths = random.sample(paths, min(max_n, len(paths)))
    imgs, bins, spoofs = [], [], []
    for p in paths:
        img = load_img(p)
        if img is not None:
            imgs.append(img)
            bins.append(label_bin)
            spoofs.append(label_spoof)
    print(f'  로드: {len(imgs)}장 ← {d}')
    return imgs, bins, spoofs

print('=== 데이터 로드 시작 ===')

# ── Live (binary=0, spoof=0) ──────────────────────────────
# 웹캠 Live: LCC 200장 + 본인 51장 = 최대 251장
lcc_imgs,  lcc_b,  lcc_s  = load_from_dir(WEBCAM_DIR, max_n=200, label_bin=0, label_spoof=0)
my_imgs,   my_b,   my_s   = load_from_dir(MY_CAM_DIR,            label_bin=0, label_spoof=0)
n_webcam = len(lcc_imgs) + len(my_imgs)

# CelebA Live: 웹캠과 동수로 샘플링 (1:1 균형)
cel_imgs,  cel_b,  cel_s  = load_from_dir(f'{CROP_DIR}/live', max_n=n_webcam, label_bin=0, label_spoof=0)

# ── Spoof (binary=1) ──────────────────────────────────────
prt_imgs,  prt_b,  prt_s  = load_from_dir(f'{CROP_DIR}/print',  max_n=300, label_bin=1, label_spoof=1)
rpl_imgs,  rpl_b,  rpl_s  = load_from_dir(f'{CROP_DIR}/replay', max_n=300, label_bin=1, label_spoof=2)
msk_imgs,  msk_b,  msk_s  = load_from_dir(f'{CROP_DIR}/mask',   max_n=300, label_bin=1, label_spoof=3)

# ── 합치기 ────────────────────────────────────────────────
all_imgs   = lcc_imgs + my_imgs + cel_imgs + prt_imgs + rpl_imgs + msk_imgs
all_binary = lcc_b    + my_b    + cel_b    + prt_b    + rpl_b    + msk_b
all_spoof  = lcc_s    + my_s    + cel_s    + prt_s    + rpl_s    + msk_s

X = np.array(all_imgs,   dtype=np.float32)
y_bin  = np.array(all_binary, dtype=np.float32)
y_sp   = np.array(all_spoof,  dtype=np.int32)

print(f'\n=== 전체 데이터 ===')
print(f'  총 이미지: {len(X)}장')
print(f'  Live(0):  {(y_bin==0).sum()}장  (웹캠 {n_webcam} + CelebA {(cel_b).count(0)})')
print(f'  Fake(1):  {(y_bin==1).sum()}장')

# ── Train / Val / Test 분할 (7:1.5:1.5) ───────────────────
idx = list(range(len(X)))
tr_idx, tmp_idx = train_test_split(idx, test_size=0.3, random_state=42, stratify=y_bin)
va_idx, te_idx  = train_test_split(tmp_idx, test_size=0.5, random_state=42, stratify=y_bin[tmp_idx])

X_tr, y_bin_tr, y_sp_tr = X[tr_idx], y_bin[tr_idx], y_sp[tr_idx]
X_va, y_bin_va, y_sp_va = X[va_idx], y_bin[va_idx], y_sp[va_idx]
X_te, y_bin_te, y_sp_te = X[te_idx], y_bin[te_idx], y_sp[te_idx]

print(f'\n=== 분할 결과 ===')
print(f'  Train: {len(X_tr)}장')
print(f'  Val:   {len(X_va)}장')
print(f'  Test:  {len(X_te)}장')

## Cell 3 — 데이터 분포 시각화

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
split_names = ['Train', 'Val', 'Test']
y_bins_all  = [y_bin_tr, y_bin_va, y_bin_te]

for ax, name, yb in zip(axes, split_names, y_bins_all):
    counts = [(yb==0).sum(), (yb==1).sum()]
    bars = ax.bar(['Live(REAL)', 'Spoof(FAKE)'], counts,
                  color=['steelblue', 'tomato'], alpha=0.8)
    for bar, c in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                str(c), ha='center', va='bottom', fontsize=11)
    ax.set_title(f'{name} ({len(yb)} total)', fontsize=12)
    ax.set_ylim(0, max(counts) * 1.2)

fig.suptitle('Fine-tuning v2 Data Distribution', fontsize=14)
plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/16_data_distribution.png', dpi=150)
plt.show()

# 샘플 이미지 확인
fig2, axes2 = plt.subplots(2, 6, figsize=(16, 6))
fig2.suptitle('Sample Images (top: webcam LCC | bottom: CelebA)', fontsize=12)
for i, ax in enumerate(axes2[0]):
    ax.imshow(lcc_imgs[i])
    ax.set_title('LCC Live', fontsize=8)
    ax.axis('off')
for i, ax in enumerate(axes2[1]):
    ax.imshow(cel_imgs[i])
    ax.set_title('CelebA Live', fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Cell 4 — Phase A: Head만 재학습 (backbone 완전 동결)

> **목적:** 새 도메인(웹캠)을 head 레벨에서 먼저 흡수  
> backbone은 건드리지 않아서 spoof head 안전 유지  
> LR 높게 (1e-3), 빠르게 수렴

In [ ]:
import tensorflow as tf

# ── 모델 로드 ─────────────────────────────────────────────
# stage2_best.h5 (원본) 또는 stage2_webcam.h5 (기존 파인튜닝)
# → 원본에서 시작하는 게 더 안정적
MODEL_PATH = f'{MODEL_DIR}/stage2_best.h5'
model = tf.keras.models.load_model(MODEL_PATH)
print('✅ 모델 로드:', MODEL_PATH)

# 레이어 이름 확인
print('\n=== 출력 레이어 확인 ===')
for layer in model.layers[-6:]:
    print(f'  {layer.name:<30} trainable={layer.trainable}')

# ── Phase A: 전체 동결 ────────────────────────────────────
model.trainable = True
for layer in model.layers:
    layer.trainable = False

# binary / spoof head만 해동
for layer in model.layers:
    if layer.name in ['binary', 'spoof', 'dense', 'dense_1',
                      'global_average_pooling2d', 'dropout']:
        layer.trainable = True

trainable = sum(1 for l in model.layers if l.trainable)
print(f'\nPhase A 학습 가능 레이어: {trainable}/{len(model.layers)}')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss={
        'binary': 'binary_crossentropy',
        'spoof' : 'sparse_categorical_crossentropy'
    },
    loss_weights={'binary': 0.7, 'spoof': 0.3},
    metrics={'binary': ['accuracy'], 'spoof': ['accuracy']}
)

cb_A = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_binary_accuracy', patience=3,
        restore_best_weights=True, mode='max'
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_binary_accuracy', factor=0.5,
        patience=2, min_lr=1e-5, verbose=1
    )
]

print('\n=== Phase A: Head 학습 시작 ===')
hist_A = model.fit(
    X_tr, {'binary': y_bin_tr, 'spoof': y_sp_tr},
    validation_data=(X_va, {'binary': y_bin_va, 'spoof': y_sp_va}),
    epochs=8,
    batch_size=32,
    callbacks=cb_A,
    verbose=1
)

print('\n✅ Phase A 완료')
print(f'  최종 val_binary_accuracy: {max(hist_A.history["val_binary_accuracy"]):.4f}')
print(f'  최종 val_spoof_accuracy:  {max(hist_A.history["val_spoof_accuracy"]):.4f}')

## Cell 5 — Phase B: 상위 15% Unfreeze Fine-tuning

> **기존 시도 3과의 차이:**
> - 기존: 상위 30% unfreeze → backbone 과도한 변화 → spoof head 과적합
> - 이번: 상위 **15%** unfreeze → 최소한의 backbone 조정
> - LR도 1e-4 → **5e-5**로 더 낮게

In [ ]:
# ── backbone 탐색 ─────────────────────────────────────────
backbone = None
for layer in model.layers:
    if 'mobilenetv2' in layer.name.lower():
        backbone = layer
        break

if backbone is None:
    print('⚠️ MobileNetV2 레이어 못 찾음 — 레이어 목록:')
    for l in model.layers:
        print(' ', l.name)
else:
    backbone.trainable = True
    n_layers = len(backbone.layers)
    freeze_until = int(n_layers * 0.85)   # 하위 85% 동결, 상위 15%만 해동

    for layer in backbone.layers[:freeze_until]:
        layer.trainable = False
    for layer in backbone.layers[freeze_until:]:
        layer.trainable = True

    frozen   = sum(1 for l in backbone.layers if not l.trainable)
    unfrozen = sum(1 for l in backbone.layers if l.trainable)
    print(f'backbone 총 레이어: {n_layers}')
    print(f'  동결: {frozen}개 (하위 85%)')
    print(f'  해동: {unfrozen}개 (상위 15%) ← 기존 30%에서 절반으로 줄임')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5),  # 기존 1e-4 → 절반
    loss={
        'binary': 'binary_crossentropy',
        'spoof' : 'sparse_categorical_crossentropy'
    },
    loss_weights={'binary': 0.7, 'spoof': 0.3},
    metrics={'binary': ['accuracy'], 'spoof': ['accuracy']}
)

cb_B = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_binary_accuracy', patience=4,
        restore_best_weights=True, mode='max'
    ),
    tf.keras.callbacks.ModelCheckpoint(
        f'{MODEL_DIR}/stage2_webcam_v2.h5',
        monitor='val_binary_accuracy',
        save_best_only=True, verbose=1, mode='max'
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_binary_accuracy', factor=0.5,
        patience=2, min_lr=1e-6, verbose=1
    )
]

print('\n=== Phase B: Fine-tuning 시작 (LR=5e-5, unfreeze 15%) ===')
hist_B = model.fit(
    X_tr, {'binary': y_bin_tr, 'spoof': y_sp_tr},
    validation_data=(X_va, {'binary': y_bin_va, 'spoof': y_sp_va}),
    epochs=20,
    batch_size=32,
    callbacks=cb_B,
    verbose=1
)

print('\n✅ Phase B 완료')
print(f'  최고 val_binary_accuracy: {max(hist_B.history["val_binary_accuracy"]):.4f}')
print(f'  최고 val_spoof_accuracy:  {max(hist_B.history["val_spoof_accuracy"]):.4f}')

## Cell 6 — 학습 곡선 & 카테고리별 검증

> **합격 기준 (전부 통과해야 배포 가능)**
> - 웹캠 Live → REAL ≥ 95%
> - CelebA Live → REAL ≥ 85%
> - Print/Replay/Mask → FAKE ≥ 95%

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ── 학습 곡선 ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Phase A + B 연결
ep_A = len(hist_A.history['binary_accuracy'])
ep_B = len(hist_B.history['binary_accuracy'])
epochs_A = range(1, ep_A + 1)
epochs_B = range(ep_A + 1, ep_A + ep_B + 1)

for ax, key, title in zip(axes,
    ['binary_accuracy', 'spoof_accuracy'],
    ['Binary Accuracy', 'Spoof Type Accuracy']):
    ax.plot(epochs_A, hist_A.history[key],         'b-',  label='Train A')
    ax.plot(epochs_A, hist_A.history[f'val_{key}'],'b--', label='Val A')
    ax.plot(epochs_B, hist_B.history[key],         'r-',  label='Train B')
    ax.plot(epochs_B, hist_B.history[f'val_{key}'],'r--', label='Val B')
    ax.axvline(ep_A, color='gray', ls=':', lw=1.5, label='A→B 전환')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.legend(fontsize=8)
    ax.set_ylim(0, 1.05)
    ax.grid(alpha=0.3)

plt.suptitle('Fine-tuning v2 학습 곡선', fontsize=13)
plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/16_training_curve.png', dpi=150)
plt.show()

# ── 카테고리별 상세 검증 ──────────────────────────────────
print('\n=== 카테고리별 검증 (Test Set) ===')

# 테스트 데이터를 카테고리별로 다시 로드 (정확한 분리를 위해)
THRESHOLD = 0.5
categories = {
    'webcam_LCC'    : (WEBCAM_DIR,              0, 0, 100),
    'webcam_my'     : (MY_CAM_DIR,              0, 0, None),
    'CelebA_live'   : (f'{CROP_DIR}/live',      0, 0, 100),
    'CelebA_print'  : (f'{CROP_DIR}/print',     1, 1, 100),
    'CelebA_replay' : (f'{CROP_DIR}/replay',    1, 2, 100),
    'CelebA_mask'   : (f'{CROP_DIR}/mask',      1, 3, 100),
}

PASS_CRITERIA = {
    'webcam_LCC'   : ('REAL', 0.95),
    'webcam_my'    : ('REAL', 0.95),
    'CelebA_live'  : ('REAL', 0.85),
    'CelebA_print' : ('FAKE', 0.95),
    'CelebA_replay': ('FAKE', 0.95),
    'CelebA_mask'  : ('FAKE', 0.90),
}

results = {}
print(f'  {"카테고리":<16} {"REAL%":>7} {"FAKE%":>7} {"기준":>8} {"판정":>6}')
print('  ' + '-' * 50)

for cat, (d, lb, ls, maxn) in categories.items():
    if not os.path.exists(d):
        print(f'  {cat:<16} → 폴더 없음, 스킵')
        continue
    imgs_cat, _, _ = load_from_dir(d, max_n=maxn, label_bin=lb, label_spoof=ls)
    if not imgs_cat:
        continue
    X_cat = np.array(imgs_cat)
    preds = model.predict(X_cat, verbose=0)
    probs = preds[0][:, 0] if isinstance(preds, list) else preds[:, 0]
    verdicts = (probs >= THRESHOLD).astype(int)  # 1=FAKE, 0=REAL

    real_pct = (verdicts == 0).mean()
    fake_pct = (verdicts == 1).mean()

    expected, threshold = PASS_CRITERIA[cat]
    actual_pct = real_pct if expected == 'REAL' else fake_pct
    passed = actual_pct >= threshold
    status = '✅ PASS' if passed else '❌ FAIL'

    results[cat] = {'real': real_pct, 'fake': fake_pct, 'passed': passed}
    print(f'  {cat:<16} {real_pct:>7.1%} {fake_pct:>7.1%} {"≥"+str(int(threshold*100))+"%":>8} {status}')

n_pass = sum(1 for r in results.values() if r['passed'])
n_total = len(results)
print(f'\n  총 {n_total}개 중 {n_pass}개 통과')
if n_pass == n_total:
    print('  🎉 전체 합격! → Cell 8에서 모델 저장')
else:
    print('  ⚠️ 일부 미달 → Cell 7 모순 확인 후 판단')

## Cell 7 — 핵심 체크: FAKE + spoof_type=Live 모순 케이스

> 발표 시연에서 치명적인 `🚨 FAKE — 유형: Live (실제 얼굴)` 모순이  
> 해결됐는지 확인. **0건이면 합격.**

In [ ]:
import numpy as np

print('=== FAKE + spoof_type=Live(0) 모순 케이스 검사 ===')
print('(웹캠 Live + CelebA Live 전체 대상)\n')

# 전체 Live 이미지 로드
live_all = []
for d in [WEBCAM_DIR, MY_CAM_DIR, f'{CROP_DIR}/live']:
    if os.path.exists(d):
        imgs, _, _ = load_from_dir(d, max_n=100, label_bin=0, label_spoof=0)
        live_all.extend(imgs)

X_live = np.array(live_all)
preds  = model.predict(X_live, batch_size=32, verbose=0)

if isinstance(preds, list) and len(preds) >= 2:
    bin_probs      = preds[0][:, 0]
    spoof_type_idx = np.argmax(preds[1], axis=1)
else:
    print('⚠️ 단일 출력 모델 — spoof_type 확인 불가')
    bin_probs      = preds[:, 0]
    spoof_type_idx = np.zeros(len(preds), dtype=int)

verdicts = (bin_probs >= 0.5).astype(int)  # 1=FAKE

# 모순: FAKE(1)인데 spoof_type=Live(0)
contradiction_mask = (verdicts == 1) & (spoof_type_idx == 0)
n_contradiction    = contradiction_mask.sum()
n_fake             = (verdicts == 1).sum()
n_total_live       = len(verdicts)

print(f'  검사 이미지:    {n_total_live}장')
print(f'  FAKE 오탐:      {n_fake}장 ({n_fake/n_total_live:.1%})')
print(f'  모순 케이스:    {n_contradiction}장 (FAKE인데 spoof_type=Live)')

if n_contradiction == 0:
    print('\n  ✅ 모순 0건 — 발표 시연 안전!')
else:
    print(f'\n  ❌ 모순 {n_contradiction}건 존재 — A안(UI 보정) 병행 필요')
    print('  → xai_explainer.py에 아래 코드 추가:')
    print('    if verdict == "FAKE" and spoof_type_idx == 0:')
    print('        spoof_type_idx = 99')
    print('        spoof_type_name = "복합 패턴 (도메인 불일치)"')

# 전체 요약
print('\n=== 최종 요약 ===')
real_rate = (verdicts == 0).mean()
print(f'  Live → REAL 판정률: {real_rate:.1%}')

SPOOF_NAMES = {0: 'Live', 1: 'Print', 2: 'Replay', 3: 'Mask'}
print('  spoof_type 분포 (FAKE 판정된 것들):')
for idx_val, name in SPOOF_NAMES.items():
    cnt = ((verdicts == 1) & (spoof_type_idx == idx_val)).sum()
    if cnt > 0:
        print(f'    {name}: {cnt}건')

## Cell 8 — 모델 저장 & 최종 요약

In [ ]:
import os

# ── 저장 ──────────────────────────────────────────────────
SAVE_PATH = f'{MODEL_DIR}/stage2_webcam_v2.h5'
model.save(SAVE_PATH)
size_mb = os.path.getsize(SAVE_PATH) / 1024 / 1024
print(f'✅ 모델 저장: {SAVE_PATH}  ({size_mb:.1f} MB)')

# ── xai_explainer.py 수정 안내 ────────────────────────────
print()
print('=== xai_explainer.py 모델 경로 수정 필요 ===')
print('  변경 전: MODEL_PATH = ".../stage2_webcam.h5"')
print('  변경 후: MODEL_PATH = ".../stage2_webcam_v2.h5"')

# ── 트러블슈팅 기록 ───────────────────────────────────────
print()
print('=== 트러블슈팅 기록 (노트북 완료) ===')
print('  TS-07: 웹캠 도메인 갭')
print('  원인: 웹캠 후처리(샤프닝) → 이미지 패턴이 CelebA Live와 달라 오탐')
print('  해결: LCC FASD Real 200장 + 본인 51장 혼합, unfreeze 15% Fine-tuning')
print('  결과: → Cell 6 결과 참조')

print()
print('=== 모델 파일 현황 ===')
for f in sorted(os.listdir(MODEL_DIR)):
    if f.endswith('.h5'):
        s = os.path.getsize(f'{MODEL_DIR}/{f}') / 1024 / 1024
        tag = ' ← 현재 최선' if 'webcam_v2' in f else ''
        print(f'  {f:<35} {s:.1f} MB{tag}')